![Cloud-First](../image/CloudFirst.png)

# SIT742: Modern Data Science
**(Module 04: Data Manipulation)**

**Session 4A: Data Wrangling II**

---

- Materials in this module have been developed to support practical learning in modern data science, big data processing, and applied analytics.
- Materials may include adapted or referenced open-source resources. Keep attribution and licence notes where applicable.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find any issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This notebook is one component of the practical and self-learning materials for Module 04.</td>
</tr>
<tr>
<td align="left">Estimated duration</td>
<td>Approximately 40 minutes, based on 240 minutes of M04 practical/self-learning work divided across six M04 notebooks.</td>
</tr>
<tr>
<td align="left">Main packages</td>
<td><code>pandas</code>, plus Python standard-library path handling.</td>
</tr>
<tr>
<td align="left">Data files</td>
<td><code>user-raw1.csv</code> and <code>user-raw2.csv</code> from the public <code>Jupyter/data/</code> folder.</td>
</tr>
</tbody>
</table>

</div>

---

**Table of Contents**

- [1. Overview and Learning Goals](#1-overview-and-learning-goals)
- [2. Setup and Data Files](#2-setup-and-data-files)
- [3. Importing CSV Data](#3-importing-csv-data)
- [4. Structuring Data](#4-structuring-data)
- [5. Data Cleaning](#5-data-cleaning)
- [6. Data Transformation](#6-data-transformation)
- [7. Student Tasks](#7-student-tasks)
- [8. Checks and Reflection](#8-checks-and-reflection)
- [9. Reflection and References](#9-reflection-and-references)


<a id="1-overview-and-learning-goals"></a>

### 1. Overview and Learning Goals

This session practises a common data wrangling workflow with two small user-data CSV files. You will load raw data, make column names and values consistent, handle missing and duplicate records, merge data frames, scale a numeric feature, and export a cleaned result.

By the end of this practical, you should be able to:

1. load CSV files into pandas `DataFrame` objects;
2. rename columns and replace coded values with readable labels;
3. remove irrelevant columns and handle missing rows;
4. merge compatible data frames and inspect duplicate records;
5. apply min-max scaling and z-score standardisation to a numeric column;
6. export a cleaned data frame without writing uncontrolled files into the notebook folder.


<a id="2-setup-and-data-files"></a>

### 2. Setup and Data Files

The required CSV files are public SIT742 data files. Choose the execution option that matches where you are running the notebook.

#### Option A: Google Colab / online execution

Use this option when you are running the notebook in Google Colab or another online notebook environment, or when you do not have the SIT742 repository cloned locally. The following code downloads the required CSV files from the public SIT742 GitHub repository into the notebook runtime and reads them with pandas.

#### Option B: Local repository execution

Use this option only if you have cloned the SIT742 repository locally and are running the notebook from its original folder structure. The file paths below are relative to this notebook location.

Keep `EXECUTION_MODE = "online"` for Option A. Change it to `"local"` only for Option B.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import sys
import tempfile

import pandas as pd

PUBLIC_DATA_BASE_URL = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
EXECUTION_MODE = "online"  # Use "online" for Google Colab; use "local" for a cloned SIT742 repository.

required_files = ["user-raw1.csv", "user-raw2.csv"]
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="sit742_m04a_output_"))


def download_public_data(required_files):
    data_dir = Path(tempfile.mkdtemp(prefix="sit742_m04a_data_"))
    downloaded_paths = {}
    for filename in required_files:
        url = f"{PUBLIC_DATA_BASE_URL}/{filename}"
        local_file = data_dir / filename
        urlretrieve(url, local_file)
        downloaded_paths[filename] = local_file
    return data_dir, downloaded_paths


def find_local_data_dir(required_files):
    candidates = [
        Path.cwd() / "Jupyter" / "data",
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd().parent.parent / "Jupyter" / "data",
    ]
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required_files):
            return candidate
    searched = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(
        "Could not find the SIT742 public data folder. Searched:\n" + searched
    )


if EXECUTION_MODE == "online":
    DATA_DIR, data_paths_by_file = download_public_data(required_files)
elif EXECUTION_MODE == "local":
    DATA_DIR = find_local_data_dir(required_files)
    data_paths_by_file = {filename: DATA_DIR / filename for filename in required_files}
else:
    raise ValueError('EXECUTION_MODE must be "online" or "local"')

data_paths = {
    "user_raw1": data_paths_by_file["user-raw1.csv"],
    "user_raw2": data_paths_by_file["user-raw2.csv"],
}

# Backward-compatible file-path variables for students comparing with older versions.
DataSet1 = data_paths["user_raw1"]
DataSet2 = data_paths["user_raw2"]

print("Python version:", sys.version.split()[0])
print("pandas version:", pd.__version__)
print("Execution mode:", EXECUTION_MODE)
print("Data folder:", DATA_DIR)
print("Temporary output folder:", OUTPUT_DIR)


No notebook-level package installation is required for this practical if you are using the SIT742 environment or Google Colab. If `pandas` is unavailable in another environment, install it before running this notebook.


Run the setup cell above, then confirm that both required CSV files are available before importing them with pandas.


In [ ]:
for name, path in data_paths.items():
    print(f"{name}: {path.name} ({path.stat().st_size} bytes)")


<a id="3-importing-csv-data"></a>

### 3. Importing CSV Data


In [ ]:
userdf1 = pd.read_csv(data_paths["user_raw1"], skipinitialspace=True)
userdf2 = pd.read_csv(data_paths["user_raw2"], skipinitialspace=True)

print(userdf1.head())
print(userdf2.head())


<a id="4-structuring-data"></a>

### 4. Structuring Data


#### 4.1 Renaming Column Names as Per Convenience


In [ ]:
new_name = {'Sex': 'Gender',
           'Addr.': 'Address'}

userdf1.rename(columns = new_name, inplace = True)

new_name = {'Surname': 'Family Name',
           'First Name': 'Given Name',
           'Addr.': 'Address'}

userdf2.rename(columns = new_name, inplace = True)

#### 4.2 Replacing the value of the rows if needed


In [ ]:
# Replace values for the column Gender
replace_values = {'M': 'Male', 'F': 'Female'}

userdf1 = userdf1.replace({'Gender': replace_values})
userdf2 = userdf2.replace({'Gender': replace_values})


#### 4.3 Removing one measurement for one attribute


In [ ]:
# Replace height in meters instead of centimeters for the column Height
userdf2["Height"] = userdf2["Height"] / 100

userdf2

#### 4.4 Exercises


In [ ]:
# Replace values for the column Address

<details><summary><u><b><font color="Blue">Click here for a practice solution</font></b></u></summary>

```python
replace_values = {'VIC': 'Victoria', 'NSW': 'New South Wales',
                  'SA': 'South Australia', 'QLD': 'Queensland',
                  'TAS': 'Tasmania'}

userdf1 = userdf1.replace({'Address': replace_values})
userdf2 = userdf2.replace({'Address': replace_values})

userdf1.head()
userdf2.head()
```

</details>


<a id="5-data-cleaning"></a>

### 5. Data Cleaning


#### 5.1 Removing the Irrelevant Columns


In [ ]:
userdf1.head()

In [ ]:
to_drop = ['PID', 'Address']

userdf1.drop(to_drop, inplace=True, axis = 1)
userdf1.head()

#### 5.2 Missing Data


In [ ]:
# find the missing value in the dataframes
# apply isnull() to the dataframes
userdf1.isnull()
userdf2.isnull()

#apply isna().any() to the dataframes 
userdf1.isna().any()
userdf2.isna().any()

# drop those null value rows in userdf1
userdf1.dropna(axis=0, inplace=True)

#drop null value columns in userdf2
userdf2.dropna(axis=1, inplace=True)

In [ ]:
userdf1.head()

In [ ]:
userdf2.head()

#### 5.3 Data Merging


In [ ]:
userdf = pd.merge(userdf1, userdf2, how='inner', on=None, left_on=None, 
                  right_on=None, left_index=False, right_index=False, 
                  sort=False, suffixes=('_x', '_y'), copy=True, indicator=False, 
                  validate=None)

#### 5.4 De-Duplicate


In [ ]:
userdf.duplicated()

This function also provides bool values for duplicate values in the dataset. 

If a dataset contains duplicate values it can be removed using the `drop_duplicates()` function. Following is the syntax of this function:

In [ ]:
userdf.drop_duplicates(subset=None, keep='first', inplace=False, ignore_index=False)

<a id="6-data-transformation"></a>

### 6. Data Transformation


#### 6.1 Using The min-max normalization


In [ ]:
# copy the data
df_min_max_scaled = userdf.copy()
  
# apply normalization techniques
df_min_max_scaled['Height'] = (df_min_max_scaled['Height'] - df_min_max_scaled['Height'].min()) / (df_min_max_scaled['Height'].max() - df_min_max_scaled['Height'].min())

In [ ]:
# view normalized data
print(df_min_max_scaled)

#### 6.2 Z-Score Standardization


In [ ]:
# copy the data
df_z_scaled = userdf.copy()
  
# apply normalization techniques
df_z_scaled['Height'] = (df_z_scaled['Height'] - df_z_scaled['Height'].mean()) / df_z_scaled['Height'].std()    
  

In [ ]:
# view normalized data   
display(df_z_scaled)

#### 6.3 Export Dataset


In [ ]:
cleaned_output_path = OUTPUT_DIR / "cleaned_user_data.csv"
df_z_scaled.to_csv(cleaned_output_path, index=False)
print(f"Cleaned data exported to: {cleaned_output_path}")


<a id="7-student-tasks"></a>

### 7. Student Tasks

Use these tasks to turn the worked wrangling examples into your own small data-preparation decisions. Keep notes on the choices you make, because there can be more than one reasonable cleaning workflow.

1. Inspect the two raw user data tables and identify at least three data-quality issues before applying any cleaning step.
2. Choose one column-renaming decision and explain why the new name is easier to work with in code.
3. After merging the user data tables, check whether the merged table has unexpected duplicate records.
4. Apply both min-max scaling and z-score standardisation to one numeric column, then describe when each transformed value might be easier to interpret.


In [ ]:
# Student task workspace for M04A.
# Add your exploratory wrangling code below. Keep each step small and inspect the DataFrame after each change.

# Example prompts:
# - Which columns have missing values?
# - Which columns should be renamed before merging?
# - Which records become duplicates after the merge?
# - Which numeric column is most useful for comparing min-max scaling with z-score standardisation?


<a id="8-checks-and-reflection"></a>

### 8. Checks and Reflection

Run the cell below after completing the notebook to confirm the main data-wrangling objects and generated output path are available.


In [ ]:
assert not userdf1.empty, "userdf1 should contain rows"
assert not userdf2.empty, "userdf2 should contain rows"
assert not userdf.empty, "The merged data frame should contain rows"
assert "Height" in df_z_scaled.columns, "The scaled data frame should include Height"
assert cleaned_output_path.exists(), "The cleaned CSV should have been exported"

print("M04A checks passed.")


<a id="9-reflection-and-references"></a>

### 9. Reflection and References

Before moving on, consider these questions:

1. Which cleaning step changed the number of rows in the data?
2. Which transformation changed the unit or scale of a variable?
3. Why is it safer to write generated files to a controlled output folder?

References:

- pandas documentation: https://pandas.pydata.org/docs/
- pandas merge documentation: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html
- scikit-learn preprocessing guide: https://scikit-learn.org/stable/modules/preprocessing.html
